In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Monitorare o annullare un job

Visualizza l'elenco dei tuoi workload nella [pagina Workloads](https://quantum.cloud.ibm.com/workloads).

## Visualizzare lo stato di un job
Vai alla [tabella Workloads](https://quantum.cloud.ibm.com/workloads) e controlla nella colonna Status se un job è stato completato o ha restituito un errore.

## Visualizzare l'utilizzo residuo
Vai alla [tabella Instances](https://quantum.cloud.ibm.com/instances) e seleziona la scheda associata al piano di cui vuoi visualizzare l'utilizzo residuo. Vengono mostrati il tempo totale utilizzato e il tempo totale rimanente nel tuo piano.

## Visualizzare le metriche sul numero di job e workload inviati
Vai alla [pagina Analytics](https://quantum.cloud.ibm.com/analytics) per vedere il numero totale di job inviati, nonché il conteggio dei workload batch e dei workload di sessione. Nota che puoi visualizzare la pagina Analytics solo per gli account che possiedi o gestisci.

## Monitorare un job
Usa l'istanza del job per controllarne lo stato o recuperarne i risultati chiamando il comando appropriato:

|                               |                                                                                                                                                                                                                                    |
| ----------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| job.result()                  | Consulta i risultati del job immediatamente dopo il suo completamento. I risultati sono disponibili solo una volta che il job è terminato. Pertanto, `job.result()` è una chiamata bloccante fino al completamento del job.        |
| job.job\_id()                 | Restituisce l'ID che identifica univocamente il job. Per recuperare i risultati in un secondo momento è necessario l'ID del job. Si consiglia quindi di salvare gli ID dei job che potresti voler recuperare in seguito.            |
| job.status()                  | Controlla lo stato del job.                                                                                                                                                                                                        |
| job = service.job(\<job\_id>) | Recupera un job inviato in precedenza. Questa chiamata richiede l'ID del job.                                                                                                                                                      |

<span id="retrieve-later"></span>
## Recuperare i risultati di un job in un secondo momento
Chiama `service.job(\<job\_id>)` per recuperare un job inviato in precedenza. Se non hai l'ID del job, o se vuoi recuperare più job contemporaneamente — inclusi job eseguiti su QPU (unità di elaborazione quantistica) in pensione — chiama invece `service.jobs()` con filtri opzionali. Consulta [QiskitRuntimeService.jobs](../api/qiskit-ibm-runtime/qiskit-runtime-service#jobs).

> **Note:** `service.jobs()` restituisce anche i job eseguiti con il pacchetto deprecato `qiskit-ibm-provider`. I job inviati con il pacchetto ancora più datato (anch'esso deprecato) `qiskit-ibmq-provider` non sono più disponibili.

### Esempio
Questo esempio restituisce i 10 job runtime più recenti eseguiti su `my_backend`:

In [ ]:
# This cell is hidden from users
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
import numpy as np


my_backend = "ibm_torino"
service = QiskitRuntimeService()
# backend = service.backend(my_backend)
backend = service.least_busy()

# Define two circuits, each with one parameter with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure_all()


pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)

params = np.random.uniform(size=(2, 3)).T

sampler_pub = (transpiled_circuit, params)

# Instantiate the new Estimator object, then run the transpiled circuit
# using the set of parameters and observables.
sampler = SamplerV2(mode=backend)
job = sampler.run([sampler_pub], shots=4)
print(job.job_id())

d305ck0ocacs73ajagvg


In [2]:
result = job.result()


spans = job.result().metadata["execution"]["execution_spans"]
print(spans)

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])


In [3]:
params = np.random.uniform(size=(2, 3))
params

array([[0.2260416 , 0.8747859 , 0.44361995],
       [0.94700856, 0.96826017, 0.98426562]])

In [4]:
mask = spans[0].mask(0)
mask

array([[[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]]])

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Initialize the account first.
service = QiskitRuntimeService()
# Use `limit` to retrieve a specific number of jobs. The default `limit` is 10.
service.jobs(backend_name=my_backend)

## Cancel a job

You can cancel a job from the IBM Quantum Platform dashboard either on the Workloads page or the details page for a specific workload. On the Workloads page, click the overflow menu at the end of the row for that workload, and select Cancel. If you are on the details page for a specific workload, use the Actions dropdown at the top of the page, and select Cancel.

In Qiskit, use `job.cancel()` to cancel a job.

## Next steps

<Admonition type="tip" title="Recommendations">
    - Try the [Grover's algorithm](/docs/tutorials/grovers-algorithm) tutorial.
    - Learn more about [Sampler execution spans](/docs/guides/sampler-input-output#execution-spans)
</Admonition>